In [ ]:
# neuron_analysis.py
# Analyzes 3D neuron morphologies from SWC files, computing features and saving to CSV.

import os
import math
import numpy as np
import networkx as nx
from scipy.spatial import ConvexHull
from scipy import linalg
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from source.feature_3D import read_swc, add_ball, swc_to_binary_volume, compute_fractal_dimension, compute_ot_distance_1d, upper_triangular, average_bending_energy_curve

# Define paths and constants
core_path = '../neuron_data/'
folders = ['Basal_ganglia', 'Cerebellum', 'Hippocampus', 'Main_olfactory_bulb', 'Retina']
label_map = {folder: idx for idx, folder in enumerate(folders)}

# Initialize feature dictionaries
feature_data = {
    'number_of_branches': {folder: [] for folder in folders},
    'avg_branch_length': {folder: [] for folder in folders},
    'edge_density': {folder: [] for folder in folders},
    'mean_betweenness_centrality': {folder: [] for folder in folders},
    'diameter': {folder: [] for folder in folders},
    'average_path_length': {folder: [] for folder in folders},
    'assortativity': {folder: [] for folder in folders},
    'spectral_entropy': {folder: [] for folder in folders},
    'circuity': {folder: [] for folder in folders},
    'mass_quartile_1': {folder: [] for folder in folders},
    'mass_quartile_2': {folder: [] for folder in folders},
    'mass_quartile_3': {folder: [] for folder in folders},
    'mass_quartile_4': {folder: [] for folder in folders},
    'directional_std_dev': {folder: [] for folder in folders},
    'weighted_entropy': {folder: [] for folder in folders},
    'orientation_order': {folder: [] for folder in folders},
    'fractal_dimension': {folder: [] for folder in folders},
    'algebraic_connectivity': {folder: [] for folder in folders},
    'sum_avg_bending_energy': {folder: [] for folder in folders}
}
id_data = {folder: [] for folder in folders}
interval_labels = [r'[0, \arccos(3/4)]', r'(\arccos(3/4), \arccos(1/2]]', r'(\arccos(1/2), \arccos(1/4]]', r'(\arccos(1/4), \pi/2]']

# Dictionary to store results
results = {folder: {} for folder in folders}

# Process each neuron
for folder in folders:
    folder_path = os.path.join(core_path, folder, 'CNG version')
    if not os.path.exists(folder_path):
        continue
    for file_name in os.listdir(folder_path):
        if file_name.endswith(".swc"):
            swc_file = os.path.join(folder_path, file_name)
            points = {}
            nodes = {}
            parents = {}
            children = {}
            try:
                # Read SWC file and filter node types
                neuron_name = os.path.basename(swc_file).split('.')[0]
                valid_types = [3] if folder == 'Cerebellum' else [1, 3]
                with open(swc_file, 'r') as f:
                    for line in f:
                        if line.startswith('#') or not line.strip():
                            continue
                        parts = line.split()
                        if len(parts) == 7:
                            index, type_, x, y, z, radius, parent = map(float, parts)
                            type_ = int(type_)
                            if type_ in valid_types or (folder == 'Retina' and type_ != 2):
                                points[index] = (index, type_, x, y, z, radius, parent)
                                nodes[index] = np.array([x, y, z])
                                parents[index] = parent
                                if parent != -1 and parent in nodes:
                                    if parent not in children:
                                        children[parent] = []
                                    children[parent].append(index)
                
                # Identify branch points, terminals, roots, and soma nodes
                branch_points = [idx for idx in nodes if idx in children and len(children[idx]) > 1]
                terminals = [idx for idx in nodes if idx not in children]
                roots = [idx for idx in nodes if parents[idx] == -1]
                soma_nodes = [idx for idx in nodes if points[idx][1] == 1]

                # Compute branch lengths
                branch_lengths = []
                visited_segments = set()
                def compute_segment_length(start_idx, end_idx):
                    length = np.linalg.norm(nodes[start_idx] - nodes[end_idx])
                    return length
                def trace_branch(start_idx, parent_idx, current_length, visited):
                    if (start_idx, parent_idx) in visited_segments or (parent_idx, start_idx) in visited_segments:
                        return None, 0.0
                    visited_segments.add((start_idx, parent_idx))
                    visited.add(start_idx)
                    path = [start_idx]
                    total_length = current_length
                    if start_idx in terminals or parents[start_idx] == -1:
                        return path, total_length
                    if start_idx in branch_points:
                        return path, total_length
                    if start_idx in children and len(children[start_idx]) == 1:
                        child_idx = children[start_idx][0]
                        if child_idx not in visited:
                            segment_length = compute_segment_length(start_idx, child_idx)
                            new_path, new_length = trace_branch(
                                child_idx, start_idx, total_length + segment_length, visited.copy()
                            )
                            if new_path:
                                return path + new_path, new_length
                    return None, 0.0

                # Identify stem nodes (nodes connected to soma)
                stem_nodes = []
                for soma_idx in soma_nodes:
                    if soma_idx in children:
                        stem_nodes.extend([c for c in children[soma_idx] if points.get(c, [0, 0])[1] != 1])

                # Start branches from stems or roots and bifurcation points
                start_points = stem_nodes if soma_nodes else roots
                start_points += branch_points

                # Trace branches from soma-to-stem segments
                for soma_idx in soma_nodes:
                    if soma_idx in children:
                        for child_idx in children[soma_idx]:
                            if points.get(child_idx, [0, 0])[1] == 1:
                                continue
                            segment_length = compute_segment_length(soma_idx, child_idx)
                            path, length = trace_branch(child_idx, soma_idx, segment_length, set())
                            if path and length > 0:
                                branch_lengths.append(length)

                # Trace branches from other start points
                for start_idx in start_points:
                    if start_idx in children:
                        for child_idx in children[start_idx]:
                            if points.get(child_idx, [0, 0])[1] == 1:
                                continue
                            segment_length = compute_segment_length(start_idx, child_idx)
                            path, length = trace_branch(child_idx, start_idx, segment_length, set())
                            if path and length > 0:
                                branch_lengths.append(length)

                # Compute branch metrics
                number_of_branches = len(branch_lengths)
                avg_branch_length = np.mean(branch_lengths) if branch_lengths else 0

                # Create networkx graph for other metrics
                G = nx.Graph()
                all_nodes = {p[0]: (p[2], p[3], p[4]) for p in points.values() if p[1] in valid_types}
                edge_lengths = []
                for p in points.values():
                    if p[1] not in valid_types:
                        continue
                    start = p[0]
                    end = p[6]
                    if end != -1 and end in all_nodes:
                        sx, sy, sz = all_nodes[start]
                        ex, ey, ez = all_nodes[end]
                        weight = math.sqrt((ex - sx)**2 + (ey - sy)**2 + (ez - sz)**2)
                        G.add_edge(start, end, weight=weight)
                        edge_lengths.append(weight)

                # Edge Density (using volume for 3D)
                all_points = np.array([(p[2], p[3], p[4]) for p in points.values() if p[1] in valid_types])
                total_edge_length = sum(edge_lengths)
                if len(np.unique(all_points, axis=0)) >= 4:
                    hull = ConvexHull(all_points)
                    volume = hull.volume
                else:
                    volume = 0
                edge_density = total_edge_length / volume if volume > 0 else 0

                # Betweenness Centrality (Mean)
                try:
                    bc = nx.betweenness_centrality(G, weight='weight', normalized=True)
                    mean_bc = np.mean(list(bc.values())) if bc else 0
                except:
                    mean_bc = 0

                # Graph Diameter
                try:
                    diameter = nx.diameter(G, weight='weight') if nx.is_connected(G) else 0
                except:
                    diameter = 0

                # Average Path Length
                try:
                    apl = nx.average_shortest_path_length(G, weight='weight') if nx.is_connected(G) else 0
                except:
                    apl = 0

                # Assortativity
                try:
                    if G.number_of_edges() < 2:
                        assortativity = 0
                    else:
                        assortativity = nx.degree_assortativity_coefficient(G, weight='weight')
                        assortativity = 0 if np.isnan(assortativity) else assortativity
                except:
                    assortativity = 0

                # Spectral Entropy
                try:
                    L = nx.laplacian_matrix(G, weight='weight').astype(float).toarray()
                    eigenvalues = linalg.eigvalsh(L)
                    eigenvalues = np.maximum(eigenvalues, 1e-12)
                    probs = eigenvalues / np.sum(eigenvalues)
                    spectral_entropy = -np.sum(probs * np.log(probs)) if probs.sum() > 0 else 0
                except:
                    spectral_entropy = 0

                # Algebraic Connectivity
                try:
                    alg_conn = nx.algebraic_connectivity(G, weight='weight', method='tracemin_pcg')
                    algebraic_connectivity = alg_conn
                except:
                    algebraic_connectivity = 0

                # Circuity
                straight_line_lengths = branch_lengths
                local_vectors = []
                local_lengths = []
                for idx, p in points.items():
                    if p[6] != -1 and p[6] in all_nodes:
                        dx = p[2] - points[p[6]][2]
                        dy = p[3] - points[p[6]][3]
                        dz = p[4] - points[p[6]][4]
                        length = math.sqrt(dx**2 + dy**2 + dz**2)
                        if length > 1e-10:
                            local_vectors.append((dx / length, dy / length, dz / length))
                            local_lengths.append(length)
                L_net = sum(local_lengths)
                L_g = sum(straight_line_lengths)
                circuity = L_net / L_g if L_g > 0 else 1

                # Fractal Dimension
                fractal_dim = compute_fractal_dimension(swc_file, folder)

                # Angular metrics (3D orientation tensor)
                mass_quartile1 = mass_quartile2 = mass_quartile3 = mass_quartile4 = 0.0
                directional_std = 0.0
                weighted_entropy = 0.0
                orientation_order = 1.0
                if local_vectors:
                    u = np.array(local_vectors)
                    edge_weights = np.array(local_lengths)
                    total_weight = np.sum(edge_weights)
                    edge_weights /= total_weight if total_weight > 0 else 1
                    T = np.sum([w * np.outer(ui, ui) for w, ui in zip(edge_weights, u)], axis=0)
                    eigvals, eigvecs = np.linalg.eigh(T)
                    lambda_max_idx = np.argmax(eigvals)
                    lambda_max = eigvals[lambda_max_idx]
                    mu = eigvecs[:, lambda_max_idx]
                    directional_variance = 1 - lambda_max
                    directional_std = np.sqrt(directional_variance)
                    dots = np.dot(u, mu)
                    u[dots < 0] = -u[dots < 0]
                    dots = np.abs(dots)
                    alphas = np.arccos(dots)
                    cos_alphas = dots
                    betas = np.zeros(len(u))
                    e1 = np.array([1., 0., 0.])
                    cross_e1_mu = np.cross(e1, mu)
                    norm_cross = np.linalg.norm(cross_e1_mu)
                    if norm_cross > 1e-10:
                        nu = cross_e1_mu / norm_cross
                    else:
                        e2 = np.array([0., 1., 0.])
                        cross_e2_mu = np.cross(e2, mu)
                        norm_cross = np.linalg.norm(cross_e2_mu)
                        if norm_cross > 1e-10:
                            nu = cross_e2_mu / norm_cross
                        else:
                            e3 = np.array([0., 0., 1.])
                            cross_e3_mu = np.cross(e3, mu)
                            nu = cross_e3_mu / np.linalg.norm(cross_e3_mu)
                    v2 = np.cross(mu, nu)
                    v1 = nu
                    for i in range(len(u)):
                        inner = dots[i]
                        proj = u[i] - inner * mu
                        proj_norm = np.linalg.norm(proj)
                        if proj_norm > 1e-10:
                            beta = np.arctan2(np.dot(proj, v2), np.dot(proj, v1))
                            if beta < 0:
                                beta += 2 * np.pi
                            betas[i] = beta
                        else:
                            betas[i] = 0.0
                    N_alpha = 8
                    N_beta = 16
                    hist_weighted, xedges, yedges = np.histogram2d(
                        cos_alphas, betas, bins=(N_alpha, N_beta),
                        range=[[0, 1], [0, 2 * np.pi]], weights=edge_weights
                    )
                    total_hist = np.sum(hist_weighted)
                    if total_hist > 0:
                        p_weighted = hist_weighted / total_hist
                    else:
                        p_weighted = np.zeros_like(hist_weighted)
                    p_weighted = p_weighted[p_weighted != 0]
                    weighted_entropy = -np.sum(p_weighted * np.log(p_weighted))
                    H_max = np.log(N_alpha * N_beta) if N_alpha * N_beta > 0 else 0
                    orientation_order = 1 - (weighted_entropy / H_max)**2 if H_max != 0 else 0
                    alpha1 = np.arccos(3/4)
                    alpha2 = np.arccos(1/2)
                    alpha3 = np.arccos(1/4)
                    mask_1 = alphas <= alpha1
                    mask_2 = (alphas > alpha1) & (alphas <= alpha2)
                    mask_3 = (alphas > alpha2) & (alphas <= alpha3)
                    mask_4 = alphas > alpha3
                    mass_quartile1 = np.sum(edge_weights * mask_1)
                    mass_quartile2 = np.sum(edge_weights * mask_2)
                    mass_quartile3 = np.sum(edge_weights * mask_3)
                    mass_quartile4 = np.sum(edge_weights * mask_4)

                # Compute bending energies
                branch_avg_bendings = []
                visited_segments = set()
                def trace_branch(start_idx, parent_idx, visited):
                    if (start_idx, parent_idx) in visited_segments or (parent_idx, start_idx) in visited_segments:
                        return None
                    visited_segments.add((start_idx, parent_idx))
                    visited.add(start_idx)
                    path = [start_idx]
                    if start_idx in terminals or parents[start_idx] == -1:
                        return path
                    if start_idx in branch_points:
                        return path
                    if start_idx in children and len(children[start_idx]) == 1:
                        child_idx = children[start_idx][0]
                        if child_idx not in visited:
                            new_path = trace_branch(child_idx, start_idx, visited.copy())
                            if new_path:
                                return path + new_path
                    return None

                for soma_idx in soma_nodes:
                    if soma_idx in children:
                        for child_idx in children[soma_idx]:
                            if points.get(child_idx, [0, 0])[1] == 1:
                                continue
                            path = trace_branch(child_idx, soma_idx, set())
                            if path:
                                full_path = [soma_idx] + path
                                if len(full_path) >= 3:
                                    c = np.array([nodes[idx] for idx in full_path])
                                    try:
                                        avg_bend = average_bending_energy_curve(c, closed=False)
                                        branch_avg_bendings.append(avg_bend)
                                    except ValueError as e:
                                        print(f"Skipping branch in {neuron_name}: {e}")
                                        branch_avg_bendings.append(0.0)
                                else:
                                    branch_avg_bendings.append(0.0)

                for start_idx in start_points:
                    if start_idx in children:
                        for child_idx in children[start_idx]:
                            if points.get(child_idx, [0, 0])[1] == 1:
                                continue
                            path = trace_branch(child_idx, start_idx, set())
                            if path:
                                full_path = [start_idx] + path
                                if len(full_path) >= 3:
                                    c = np.array([nodes[idx] for idx in full_path])
                                    try:
                                        avg_bend = average_bending_energy_curve(c, closed=False)
                                        branch_avg_bendings.append(avg_bend)
                                    except ValueError as e:
                                        print(f"Skipping branch in {neuron_name}: {e}")
                                        branch_avg_bendings.append(0.0)
                                else:
                                    branch_avg_bendings.append(0.0)

                sum_avg_bending = sum(branch_avg_bendings)

                # Store results
                metrics = {
                    'number_of_branches': number_of_branches,
                    'avg_branch_length': avg_branch_length,
                    'edge_density': edge_density,
                    'mean_betweenness_centrality': mean_bc,
                    'diameter': diameter,
                    'average_path_length': apl,
                    'assortativity': assortativity,
                    'spectral_entropy': spectral_entropy,
                    'algebraic_connectivity': algebraic_connectivity,
                    'circuity': circuity,
                    'mass_quartile_1': mass_quartile1,
                    'mass_quartile_2': mass_quartile2,
                    'mass_quartile_3': mass_quartile3,
                    'mass_quartile_4': mass_quartile4,
                    'directional_std_dev': directional_std,
                    'weighted_entropy': weighted_entropy,
                    'orientation_order': orientation_order,
                    'fractal_dimension': fractal_dim,
                    'sum_avg_bending_energy': sum_avg_bending
                }
                results[folder][neuron_name] = metrics

                # Append data to feature dictionaries
                for key in feature_data:
                    feature_data[key][folder].append(metrics[key])
                id_data[folder].append(neuron_name)
            except Exception as e:
                print(f"Error processing {swc_file}: {e}")

# Compile features into DataFrame and save to CSV
data = []
for folder in folders:
    num_files = len(id_data[folder])
    for i in range(num_files):
        row = {
            'id': id_data[folder][i],
            'label': label_map[folder],
        }
        for feature in feature_data:
            row[feature] = feature_data[feature][folder][i]
        data.append(row)

df = pd.DataFrame(data)
df.to_csv('neuron_features.csv', index=False)
print("Feature data saved to neuron_features.csv")